# 探索性数据分析（EDA）

**负责人：** 3 号　**阶段：** 1 数据准备

摸清数据：粒度、质量、类别分布、文本特征，以及它们和情绪标签的关系。本 notebook 由 2 号在 11/2 代为完成，供中期报告使用；请 3 号复核，并据此写报告第 2 章。

## 任务

标"（选做）"的任务在必做部分完成后再做。

- [x] 说明数据粒度（一行 = 一条推文），整理数据字典
- [x] 统计缺失值、无效值、重复值
- [x] 类别分布和不平衡程度
- [x] 各类别文本长度（箱线图）
- [x] 高频 hashtag；各类别高频 1-gram / 2-gram（用条形图代替词云，能看出具体数量）
- [x] 子市场（加密 / 美股 / 印度）× 情绪
- [x] 相关性：文本长度、hashtag 占比与标签
- [x] 协助 2 号抽样核对 `__` 和乱码样本（清洗时已完成）

## 产出

- `figures/eda_*.png`（6 张统计图）
- 报告第 2 章要用的数字（表 3 和各图的结论）

In [ ]:
import sys
from pathlib import Path

# 向上查找仓库根目录（含 src/config.py 的文件夹），在任何子文件夹里打开都能导入 src
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists()), None)
if ROOT is None:
    raise RuntimeError("找不到仓库根目录：请在 clone 下来的仓库里启动 Jupyter（Colab 先 %cd 进仓库）")
sys.path.insert(0, str(ROOT))

from src.config import DATA_ROOT, FIGURES_DIR, LABELS, SEED                 # 路径和常量
from src.data import load_clean, load_raw, load_splits                      # 读取数据和划分
from src.evaluate import evaluate, new_run_id, plot_confusion, save_result  # 统一评分
from src.cache import load_embedding, save_embedding, save_preds            # 嵌入和预测结果的读写

print("仓库根目录:", ROOT)
print("DATA_ROOT:", DATA_ROOT)

## 1. Granularity and data dictionary

Data quality is measured on the **raw file** (1,698 tweets). The figures use the **cleaned data** `clean_v2` (1,525 tweets), because that is what every model reads and the raw text still contains broken characters.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from scipy.stats import kruskal, spearmanr

from src.config import CLEAN_DIR, CLEAN_VERSION

raw = load_raw()
full = pd.read_csv(CLEAN_DIR / f"clean_v{CLEAN_VERSION}_all.csv", keep_default_na=False)
clean = load_clean()
clean["words"] = clean["text_clean"].str.split().str.len()

print("granularity: one row = one tweet")
print("raw file columns: text, text_sentiment | rows:", len(raw), "| load_raw() renames text_sentiment to label and adds id (row number)")
pd.DataFrame({
    "field": ["text", "text_sentiment"],
    "type": ["string", "integer"],
    "meaning": ["tweet text as collected", "sentiment label: 0 Neutral, 1 Positive, 2 Negative"],
    "missing": [int(raw["text"].isna().sum()), int(raw["label"].isna().sum())],
})

## 2. Data quality (Table 3 of the report)

Invalid rows are those removed during cleaning (see `data/clean_v2/clean_v2_log.md`).

In [ ]:
n = len(raw)
reasons = full.loc[full["keep"] == 0, "drop_reason"].value_counts()
raw_words = raw["text"].str.split().str.len()


def share(k):
    return f"{k:,} ({k / n:.1%})"


quality = pd.DataFrame([
    ("Raw records", f"{n:,}", "one tweet per row, two columns"),
    ("Missing text or labels", share(int(raw.isna().any(axis=1).sum())), "nothing to impute"),
    ("Exact duplicates", share(int(reasons.get("exact_duplicate", 0))), "same text repeated; removed"),
    ("Content duplicates", share(int(reasons.get("content_duplicate", 0))), "same words, different mentions or emoji; removed"),
    ("Unreadable (damaged non-Latin text)", share(int(reasons.get("unreadable_encoding", 0))), "characters lost in the file; removed"),
    ("Not in English", share(int(reasons.get("non_english", 0) + reasons.get("non_english_manual", 0))), "labels are artefacts; removed"),
    ("No words (only hashtags or mentions)", share(int(reasons.get("no_content", 0))), "all labelled Neutral; removed"),
    ("Tweets with lost characters (emoji)", share(int(full["emoji_lost"].sum())), "flagged in emoji_lost, kept"),
    ("Median text length", f"{int(raw_words.median())} words ({int(raw['text'].str.len().median())} characters)", "short texts; max " + str(int(raw_words.max())) + " words"),
], columns=["measure", "value", "interpretation"])
quality

## 3. Figures

Every figure answers one question. Colours follow the sentiment: red Negative, grey Neutral, blue Positive.

In [ ]:
SURFACE, INK, INK_2, AXIS, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#c3c2b7", "#e1e0d9"
CLASSES = ["Negative", "Neutral", "Positive"]
SENTIMENT = {"Negative": "#e34948", "Neutral": "#c3c2b7", "Positive": "#2a78d6"}
NEUTRAL_BAR = "#3d4b5c"  # one-series bars that are not about sentiment


def new_figure(width=6.8, height=3.0, ncols=1):
    fig, axes = plt.subplots(1, ncols, figsize=(width, height), facecolor=SURFACE)
    for ax in np.atleast_1d(axes):
        ax.set_facecolor(SURFACE)
        for side in ("top", "right", "left"):
            ax.spines[side].set_visible(False)
        ax.spines["bottom"].set_color(AXIS)
        ax.tick_params(colors=INK_2, labelsize=8, length=0)
    return fig, axes


def title(ax, text, pad=6, size=10.5):
    ax.set_title(text, loc="left", fontsize=size, color=INK, pad=pad)


def stacked_shares(ax, table, labels, xticks=(0, 0.25, 0.5, 0.75, 1)):
    """100% stacked horizontal bars, one row per group, percentage labels inside wide segments."""
    left = np.zeros(len(table))
    for name in CLASSES:
        values = table[name].to_numpy()
        ax.barh(labels, values, left=left, height=0.6, color=SENTIMENT[name], edgecolor=SURFACE, linewidth=1.5, label=name)
        for y, (v, l) in enumerate(zip(values, left)):
            if v >= 0.10:
                ax.text(l + v / 2, y, f"{v:.0%}", ha="center", va="center", fontsize=7.5, color=INK)
        left += values
    ax.set_xlim(0, 1)
    ax.set_xticks(list(xticks))
    ax.invert_yaxis()
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
    ax.tick_params(axis="y", labelsize=8, colors=INK)

### Figure 1. How many tweets does each class have?

In [ ]:
after = clean["label_name"].value_counts()[CLASSES]
before = raw["label"].map(LABELS).value_counts()[CLASSES]
fig, ax = new_figure(height=2.2)
ax.barh(CLASSES, after.values, height=0.6, color=[SENTIMENT[c] for c in CLASSES])
for y, c in enumerate(CLASSES):
    ax.text(after[c] + 8, y, f"{after[c]} ({after[c] / after.sum():.1%})   raw: {before[c]}", va="center", fontsize=8, color=INK_2)
ax.invert_yaxis()
ax.set_xlim(0, after.max() * 1.55)
ax.tick_params(axis="y", labelsize=8.5, colors=INK)
ax.set_xlabel("tweets after cleaning", fontsize=8, color=INK_2)
title(ax, f"Class distribution after cleaning ({after.sum():,} tweets)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_class_distribution.png", dpi=200, facecolor=SURFACE)
plt.show()
print(f"largest / smallest class: {after.max() / after.min():.2f} after cleaning, {before.max() / before.min():.2f} before")

### Figure 2. Are Neutral tweets shorter?

In [ ]:
groups = [clean.loc[clean["label_name"] == c, "words"] for c in CLASSES]
fig, ax = new_figure(height=2.4)
box = ax.boxplot(groups, vert=False, widths=0.55, patch_artist=True, tick_labels=CLASSES,
                 medianprops={"color": INK, "linewidth": 1.5}, whiskerprops={"color": INK_2},
                 capprops={"color": INK_2}, flierprops={"marker": "o", "markersize": 2.5, "markerfacecolor": AXIS, "markeredgecolor": AXIS})
for patch, c in zip(box["boxes"], CLASSES):
    patch.set_facecolor(SENTIMENT[c])
    patch.set_edgecolor(INK_2)
for y, g in enumerate(groups, start=1):
    ax.text(g.max() + 1.5, y, f"median {int(g.median())}", va="center", fontsize=8, color=INK_2)
ax.invert_yaxis()
ax.set_xlim(0, clean["words"].max() * 1.22)
ax.tick_params(axis="y", labelsize=8.5, colors=INK)
ax.set_xlabel("words per tweet (cleaned text)", fontsize=8, color=INK_2)
title(ax, "Tweet length by class")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_text_length.png", dpi=200, facecolor=SURFACE)
plt.show()
result = kruskal(*groups)
print("median words:", {c: int(g.median()) for c, g in zip(CLASSES, groups)}, f"| Kruskal-Wallis H = {result.statistic:.1f}, p = {result.pvalue:.1e}")

### Figure 3. Which hashtags are most common?

In [ ]:
tag_counts = Counter(tag for tags in clean["hashtags"] for tag in set(tags.split()))
top_tags = pd.Series(dict(tag_counts.most_common(15)))
fig, ax = new_figure(height=3.6)
ax.barh(["#" + t for t in top_tags.index], top_tags.values, height=0.6, color=NEUTRAL_BAR)
for y, v in enumerate(top_tags.values):
    ax.text(v + 8, y, f"{v} ({v / len(clean):.0%})", va="center", fontsize=7.5, color=INK_2)
ax.invert_yaxis()
ax.set_xlim(0, top_tags.max() * 1.2)
ax.tick_params(axis="y", labelsize=8, colors=INK)
ax.set_xlabel("tweets containing the hashtag", fontsize=8, color=INK_2)
title(ax, "15 most common hashtags (share of the 1,525 tweets)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_top_hashtags.png", dpi=200, facecolor=SURFACE)
plt.show()
print("tweets with at least one crypto hashtag or ticker:", int(clean["domain_crypto"].sum()))

### Figure 4. Which words does each class use most?

Words are counted in `text_norm` (lemmatised, stopwords removed but negations and direction words kept). A word that comes from the tweet's own hashtags is not counted, otherwise *stockmarketcrash* and *bearmarket* would top every list. The value is the share of the class's tweets that contain the word.

In [ ]:
def content_words(row):
    tags = set(row["hashtags"].split())
    return [w for w in row["text_norm"].split() if w not in tags]


tokens = clean.apply(content_words, axis=1)
fig, axes = new_figure(height=3.2, ncols=3)
top_words = {}
for ax, c in zip(axes, CLASSES):
    in_class = tokens[clean["label_name"] == c]
    counts = Counter(w for ws in in_class for w in set(ws))
    top = pd.Series(dict(counts.most_common(10))) / len(in_class)
    top_words[c] = top
    ax.barh(top.index, top.values, height=0.6, color=SENTIMENT[c])
    for y, v in enumerate(top.values):
        ax.text(v + 0.005, y, f"{v:.0%}", va="center", fontsize=7, color=INK_2)
    ax.invert_yaxis()
    ax.set_xlim(0, 0.4)
    ax.set_xticks([])
    ax.spines["bottom"].set_visible(False)
    ax.tick_params(axis="y", labelsize=8, colors=INK)
    title(ax, f"{c} (n = {len(in_class)})", pad=4, size=9.5)
fig.suptitle("Most frequent words per class (share of the class's tweets)", x=0.02, ha="left", fontsize=10.5, color=INK)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_top_words.png", dpi=200, facecolor=SURFACE)
plt.show()

# Most frequent word pairs per class (for the appendix)
bigrams = {}
for c in CLASSES:
    pairs = Counter(p for ws in tokens[clean["label_name"] == c] for p in set(zip(ws, ws[1:])))
    bigrams[c] = [f"{a} {b} ({k})" for (a, b), k in pairs.most_common(8)]
pd.DataFrame(bigrams)

### Figure 5. Does the sentiment differ between markets?

Markets are detected from hashtags and cashtags (`domain_crypto`, `domain_india`, `domain_us` in `clean_v2`). A tweet can belong to more than one market.

In [ ]:
market_rows = {
    "Crypto": clean["domain_crypto"] == 1,
    "India": clean["domain_india"] == 1,
    "US": clean["domain_us"] == 1,
    "No market tag": (clean[["domain_crypto", "domain_india", "domain_us"]].sum(axis=1) == 0),
    "All tweets": pd.Series(True, index=clean.index),
}
market = pd.DataFrame({name: clean.loc[mask, "label_name"].value_counts(normalize=True) for name, mask in market_rows.items()}).T[CLASSES]
sizes = {name: int(mask.sum()) for name, mask in market_rows.items()}
fig, ax = new_figure(height=2.7)
stacked_shares(ax, market, [f"{name} (n = {sizes[name]:,})" for name in market.index])
ax.legend(ncol=3, frameon=False, loc="lower left", bbox_to_anchor=(0, 1.0), fontsize=8, labelcolor=INK_2)
title(ax, "Label share by market", pad=22)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_market_sentiment.png", dpi=200, facecolor=SURFACE)
plt.show()
market.round(3)

### Figure 6. Are length and hashtag share related to the label?

Tweets are grouped into four length bins (quartiles of words per tweet) and four bins of hashtag share (hashtags per word).

In [ ]:
length_bins, edges = pd.qcut(clean["words"], 4, retbins=True)
edges = np.round(edges).astype(int)
length_labels = [f"{edges[0] if i == 0 else edges[i] + 1}–{edges[i + 1]} words" for i in range(4)]
length_table = pd.crosstab(length_bins, clean["label_name"], normalize="index")[CLASSES]
length_sizes = length_bins.value_counts().sort_index().to_numpy()

ratio_edges = [-0.001, 0.10, 0.25, 0.50, 1.0]
ratio_names = ["≤ 10%", "10–25%", "25–50%", "> 50%"]
ratio_bins = pd.cut(clean["hashtag_ratio"], ratio_edges, labels=ratio_names)
ratio_table = pd.crosstab(ratio_bins, clean["label_name"], normalize="index")[CLASSES]
ratio_sizes = ratio_bins.value_counts().reindex(ratio_names).to_numpy()

fig, axes = new_figure(height=2.8, ncols=2)
stacked_shares(axes[0], length_table, [f"{l} (n = {s})" for l, s in zip(length_labels, length_sizes)], xticks=(0, 0.5, 1))
stacked_shares(axes[1], ratio_table, [f"{l} (n = {s})" for l, s in zip(ratio_names, ratio_sizes)], xticks=(0, 0.5, 1))
title(axes[0], "by tweet length", pad=4, size=9.5)
title(axes[1], "by share of hashtags", pad=4, size=9.5)
handles, names = axes[0].get_legend_handles_labels()
fig.suptitle("Label share by tweet length and hashtag share", x=0.02, y=0.98, ha="left", fontsize=10.5, color=INK)
fig.legend(handles, names, ncol=3, frameon=False, loc="upper left", bbox_to_anchor=(0.015, 0.93), fontsize=8, labelcolor=INK_2)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(FIGURES_DIR / "eda_length_hashtags.png", dpi=200, facecolor=SURFACE)
plt.show()

features = ["words", "n_hashtags", "hashtag_ratio", "caps_ratio", "n_exclaim", "n_question", "emoji_lost"]
pd.DataFrame({c: [spearmanr(clean[f], clean["label_name"] == c)[0] for f in features] for c in CLASSES},
             index=features).round(3).rename_axis("Spearman correlation with 'label is ...'")

## 4. Findings for Chapter 2

- **Granularity and quality.** One row is one tweet. Nothing is missing, but 173 of the 1,698 tweets (10.2%) are duplicates, unreadable, not in English or without words, and 28% lost characters such as emoji in the file.
- **Classes.** After cleaning, Positive is the largest class (39.8%) and Neutral the smallest (28.4%); the largest-to-smallest ratio of 1.40 is a mild imbalance. Macro-F1 is used as the main metric so that each class counts equally.
- **Length.** Neutral tweets are about half as long as the others (median 13 words against 25 and 26). The difference is far beyond chance (Kruskal-Wallis test). Short tweets give a sentiment lexicon few words to score, so they end up Neutral.
- **Hashtags.** The two collection hashtags dominate (#stockmarketcrash, #bearmarket); after them, crypto hashtags (#crypto, #bitcoin, #btc) are the most common topic. The more of a tweet consists of hashtags, the more likely it is Neutral: 72% of tweets where hashtags are more than half of the words are Neutral.
- **Words.** *not* is among the most frequent words in every class, which supports keeping negations in `text_norm`. Positive tweets more often use *buy*, *like*, *good* and *invest*; Negative tweets *crash*, *down* and *no*.
- **Markets.** The label mix is similar across crypto, Indian and US tweets (about 40% Positive in each), so the market alone says little about sentiment.

Together these patterns fit labels produced by a word-based sentiment tool (see `31_label_noise_errors`): what matters is which sentiment words a tweet contains, not its topic.